In [1]:
import boto3
import time

ec2 = boto3.client('ec2', region_name='ap-south-1')


# 1. Create VPC
vpc = ec2.create_vpc(CidrBlock='10.0.0.0/16')
vpc_id = vpc['Vpc']['VpcId']

ec2.modify_vpc_attribute(VpcId=vpc_id, EnableDnsSupport={'Value': True})
ec2.modify_vpc_attribute(VpcId=vpc_id, EnableDnsHostnames={'Value': True})

print("VPC Created:", vpc_id)

time.sleep(2)

# 2. Create Subnet
subnet = ec2.create_subnet(
    VpcId=vpc_id,
    CidrBlock='10.0.1.0/24',
    AvailabilityZone='ap-south-1a'
)

subnet_id = subnet['Subnet']['SubnetId']
print("Subnet Created:", subnet_id)

# Enable Auto-assign Public IP
ec2.modify_subnet_attribute(
    SubnetId=subnet_id,
    MapPublicIpOnLaunch={'Value': True}
)

# 3. Create Internet Gateway
igw = ec2.create_internet_gateway()
igw_id = igw['InternetGateway']['InternetGatewayId']

print("Internet Gateway Created:", igw_id)

# Attach IGW to VPC
ec2.attach_internet_gateway(
    InternetGatewayId=igw_id,
    VpcId=vpc_id
)

print("Internet Gateway Attached to VPC")

# 4. Create Route Table
route_table = ec2.create_route_table(VpcId=vpc_id)
route_table_id = route_table['RouteTable']['RouteTableId']

# Create route to Internet
ec2.create_route(
    RouteTableId=route_table_id,
    DestinationCidrBlock='0.0.0.0/0',
    GatewayId=igw_id
)

# Associate Route Table with Subnet
ec2.associate_route_table(
    SubnetId=subnet_id,
    RouteTableId=route_table_id
)

print("Route Table Configured")

# 5. Create Security Group
security_group = ec2.create_security_group(
    GroupName='activity-sg',
    Description='Security group for activity',
    VpcId=vpc_id
)

sg_id = security_group['GroupId']

# Allow SSH (22)
ec2.authorize_security_group_ingress(
    GroupId=sg_id,
    IpPermissions=[
        {
            'IpProtocol': 'tcp',
            'FromPort': 22,
            'ToPort': 22,
            'IpRanges': [{'CidrIp': '0.0.0.0/0'}]
        },
        {
            'IpProtocol': 'tcp',
            'FromPort': 80,
            'ToPort': 80,
            'IpRanges': [{'CidrIp': '0.0.0.0/0'}]
        }
    ]
)

VPC Created: vpc-00bc5b0468d45629c
Subnet Created: subnet-01f278652adea9bcc
Internet Gateway Created: igw-0e1ece1ba12f7975a
Internet Gateway Attached to VPC
Route Table Configured


{'Return': True,
 'SecurityGroupRules': [{'SecurityGroupRuleId': 'sgr-052015b303df86002',
   'GroupId': 'sg-0990c0b67c621e052',
   'GroupOwnerId': '837925920700',
   'IsEgress': False,
   'IpProtocol': 'tcp',
   'FromPort': 22,
   'ToPort': 22,
   'CidrIpv4': '0.0.0.0/0',
   'SecurityGroupRuleArn': 'arn:aws:ec2:ap-south-1:837925920700:security-group-rule/sgr-052015b303df86002'},
  {'SecurityGroupRuleId': 'sgr-08b54bbef380f21e1',
   'GroupId': 'sg-0990c0b67c621e052',
   'GroupOwnerId': '837925920700',
   'IsEgress': False,
   'IpProtocol': 'tcp',
   'FromPort': 80,
   'ToPort': 80,
   'CidrIpv4': '0.0.0.0/0',
   'SecurityGroupRuleArn': 'arn:aws:ec2:ap-south-1:837925920700:security-group-rule/sgr-08b54bbef380f21e1'}],
 'ResponseMetadata': {'RequestId': 'a246f508-0e62-4a50-8ea2-6eb41c826fcf',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'a246f508-0e62-4a50-8ea2-6eb41c826fcf',
   'cache-control': 'no-cache, no-store',
   'strict-transport-security': 'max-age=31536000; inc

In [5]:
import boto3

REGION = "ap-south-1"

ec2_client = boto3.client("ec2", region_name=REGION)

user_data_script = """#!/bin/bash
yum update -y
yum install -y httpd
systemctl enable httpd
systemctl start httpd
echo "<h1>Hello from Harshad - AWS SDK Activity</h1>" > /var/www/html/index.html
"""

response = ec2_client.run_instances(
    ImageId="ami-051a31ab2f4d498f5",
    InstanceType="t3.micro",
    MinCount=1,
    MaxCount=1,
    SubnetId="subnet-0ce7e62175795a708",
    UserData=user_data_script
)

instance_id = response["Instances"][0]["InstanceId"]

print("EC2 Instance Successfully Launched!")
print("Instance ID:", instance_id)

EC2 Instance Successfully Launched!
Instance ID: i-0857fdeb541dd6208


In [13]:
import boto3

REGION = "us-east-1"

s3 = boto3.client("s3", region_name=REGION)

bucket_name = f"harshad-sdk-activity-7237"

response = s3.create_bucket(
    Bucket=bucket_name
)

print("S3 Bucket Created Successfully!")
print("Bucket Name:", bucket_name)

S3 Bucket Created Successfully!
Bucket Name: harshad-sdk-activity-7237


In [14]:
import boto3

iam = boto3.client("iam")

user_name = "harshad"

response = iam.create_user(
    UserName=user_name
)

print("IAM User Created Successfully!")
print("User Name:", response["User"]["UserName"])
print("User ARN:", response["User"]["Arn"])

IAM User Created Successfully!
User Name: harshad
User ARN: arn:aws:iam::837925920700:user/harshad
